# Huấn luyện & đánh giá các model Super-Resolution

Notebook này train và evaluate các model của dự án rồi in bảng so sánh metric. Nó
gọi trực tiếp các hàm trong `src/`, không chạy qua script.

**4 model được train từ đầu trên DF2K, cùng ngân sách 20 epoch (so sánh công bằng):**
SRCNN, SR3, SRGAN, SwinIR (bản light). Cả 4 eval trên cùng DIV2K-Val crop 256.

Tùy chọn: **SwinIR (official)** — bản SwinIR-M với weights tác giả đã train hội tụ,
làm mốc SOTA tham chiếu (không cùng ngân sách). Cần tải weights bằng
`bash scripts/download_weights.sh`; nếu không có sẽ tự bỏ qua.

Trước khi chạy: chọn kernel là Python của `.venv` trong dự án.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# thêm thư mục gốc dự án vào sys.path để import được package src/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from src.utils import load_config
from src.models import build_model
from src.datasets import build_dataset
from src.metrics import build_metrics
from src.engine import Trainer, evaluate_model

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device:", DEVICE)

## 2. Cấu hình

Chỉnh các biến dưới đây để chọn model và chế độ chạy. Để trống (None) là dùng đúng
giá trị trong file config.

In [ ]:
# 4 model train từ đầu trên DF2K (cùng ngân sách 20 epoch): tên -> file config
TRAIN_MODELS = {
    "SRCNN":  "configs/srcnn_df2k_x4.yaml",
    "SR3":    "configs/sr3_df2k_x4.yaml",
    "SRGAN":  "configs/srgan_df2k_x4.yaml",
    "SwinIR": "configs/swinir_df2k_x4.yaml",
}

# Tùy chọn: mốc SOTA tham chiếu bằng weights chính thức (KHÔNG cùng ngân sách).
# Để {} nếu không cần. Tải weights trước: bash scripts/download_weights.sh
EVAL_ONLY_MODELS = {
    "SwinIR (official)": ("configs/swinir_x4.yaml",
                          "weights/001_classicalSR_DF2K_s64w8_SwinIR-M_x4.pth"),
}

# None -> dùng epochs trong config (20). Số nhỏ (vd 3) -> chạy thử nhanh.
EPOCHS_OVERRIDE = None

# None -> giữ repeat trong config (1). Không cần đổi.
REPEAT_OVERRIDE = None

# 0 -> eval toàn bộ ảnh. Đặt N (vd 10) để chỉ eval N ảnh đầu cho nhanh.
EVAL_LIMIT = 0

## 3. Hàm train và eval

In [ ]:
def _label(d):
    """Tên hiển thị của một dataset trong config (label nếu có, không thì name)."""
    return d.get("label", d.name)


def train_one(config_path, epochs=None, repeat=None):
    """Train một model theo file config, lưu checkpoint vào experiments/<name>/."""
    cfg = load_config(config_path)
    if epochs is not None:
        cfg.train["epochs"] = epochs
    if repeat is not None:
        cfg.train_dataset.args["repeat"] = repeat

    out_dir = f"experiments/{cfg.get('name', Path(config_path).stem)}"
    model = build_model(cfg.model)
    train_set = build_dataset(cfg.train_dataset)
    val_sets = {_label(d): build_dataset(d) for d in cfg.get("val_datasets", [])}
    # validation trong lúc train dùng metric nhẹ (psnr, ssim)
    metrics = build_metrics(cfg.get("val_metrics", ["psnr", "ssim"]))

    trainer = Trainer(
        model, train_set, cfg.train, device=DEVICE,
        val_sets=val_sets, metrics=metrics, out_dir=out_dir,
    )
    trainer.train()
    return out_dir


def eval_one(config_path, checkpoint=None, limit=0):
    """Đánh giá một checkpoint trên test_datasets với toàn bộ metric.
    checkpoint=None: tự lấy best.pth (không có thì last.pth) trong experiments/<name>/.
    Trả về {dataset: {metric: value}}."""
    cfg = load_config(config_path)
    if checkpoint is None:
        out = Path(f"experiments/{cfg.get('name', Path(config_path).stem)}")
        checkpoint = out / "best.pth" if (out / "best.pth").exists() else out / "last.pth"
    print("checkpoint:", checkpoint)

    model = build_model(cfg.model).to(DEVICE)
    state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
    # gỡ lớp bọc: repo lưu {"model": ...}; weights chính thức (SwinIR) dùng
    # {"params": ...} hoặc {"params_ema": ...}
    if isinstance(state, dict):
        for key in ("model", "params_ema", "params"):
            if key in state:
                state = state[key]
                break
    model.load_state_dict(state, strict=False)

    datasets = {_label(d): build_dataset(d) for d in cfg.test_datasets}
    if limit:
        from torch.utils.data import Subset
        datasets = {k: Subset(v, range(min(limit, len(v)))) for k, v in datasets.items()}
    metrics = build_metrics(cfg.get("metrics", ["psnr", "ssim", "lpips"]))
    return evaluate_model(model, datasets, metrics, DEVICE)

## 4. Huấn luyện

Chạy ô này để train lần lượt cả 4 model trong `TRAIN_MODELS` với cùng ngân sách
20 epoch. SR3 và SRGAN/SwinIR chậm và nên chạy trên GPU; muốn thử nhanh thì đặt
`EPOCHS_OVERRIDE` ở mục Cấu hình.

In [ ]:
for name, cfg_path in TRAIN_MODELS.items():
    print(f"\n========== TRAIN {name} ==========")
    train_one(cfg_path, epochs=EPOCHS_OVERRIDE, repeat=REPEAT_OVERRIDE)

## 5. Visualize quá trình training

Hình 1: training loss của từng model, mỗi model một ô riêng — vì 4 model dùng 4 hàm
loss khác nhau (Charbonnier / GAN composite / L1 trên nhiễu), thang đo khác nhau,
chỉ dùng để xem mỗi model có hội tụ hay không.

Hình 2: thước đo chung để so sánh giữa các model — validation PSNR và SSIM trên
cùng DIV2K-Val crop 256.

Nguồn: `experiments/<name>/history.json`; model chạy bằng code cũ thì parse từ
`train_<model>.log`.


In [ ]:
import json, re
import matplotlib.pyplot as plt

def load_history(cfg_path, log_path=None):
    """history.json nếu có, không thì parse log (quét tuần tự để gán val vào epoch)."""
    cfg = load_config(cfg_path)
    p = Path(f"experiments/{cfg.get('name', Path(cfg_path).stem)}/history.json")
    if p.exists():
        return json.load(open(p))
    if log_path and Path(log_path).exists():
        h = {"train_loss": [], "val": []}
        text = Path(log_path).read_text(errors="ignore").replace("\r", "\n")
        cur = None
        for line in text.splitlines():
            m = re.match(r"epoch (\d+): avg loss ([\d.]+)(?: \(lr ([\d.e+-]+)\))?", line)
            if m:
                cur = int(m.group(1))
                e = {"epoch": cur, "loss": float(m.group(2))}
                if m.group(3): e["lr"] = float(m.group(3))
                h["train_loss"].append(e)
                continue
            m = re.match(r"\[val/([^\]]+)\] (.+)", line)
            if m and cur is not None:
                scores = {k: float(v) for k, v in re.findall(r"(\w+)=([\d.]+)", m.group(2))}
                h["val"].append({"epoch": cur, m.group(1): scores})
        return h
    return None

LOGS = {name: f"train_{name.lower()}.log" for name in TRAIN_MODELS}

# Train trên tiles: 1 tile-epoch (30581 tiles) ~ 8.86 image-epoch (3450 ảnh gốc).
# Quy đổi trục x về image-epoch để so được với các run cũ chạy 20/50 epoch.
EPOCH_SCALE = 30581 / 3450  # đặt = 1 nếu run trên ảnh gốc

def ep_axis(epochs):
    return [e * EPOCH_SCALE for e in epochs]
HIST = {n: load_history(c, LOGS.get(n)) for n, c in TRAIN_MODELS.items()}

# Loss mỗi model dùng hàm loss KHÁC NHAU (Charbonnier / GAN composite / L1 trên nhiễu)
# nên vẽ riêng từng ô, thang riêng — chỉ để xem hội tụ, không so giữa các model.
fig, axes = plt.subplots(1, len(HIST), figsize=(4.2 * len(HIST), 3.2))
for ax, (name, h) in zip(axes, HIST.items()):
    if not h or not h["train_loss"]:
        ax.set_title(f"{name} (chưa có dữ liệu)"); continue
    ep = [e["epoch"] for e in h["train_loss"]]
    ax.plot(ep_axis(ep), [e["loss"] for e in h["train_loss"]], marker="o", color="tab:blue")
    ax.set_title(f"{name} — training loss"); ax.set_xlabel("image-epoch tương đương"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Thước đo CHUNG để so sánh: val PSNR / SSIM trên cùng DIV2K-Val crop 256.
def val_points(h, metric):
    return [(v["epoch"], s[metric]) for v in h["val"] if v.get("epoch")
            for ds, s in v.items() if isinstance(s, dict) and metric in s]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in HIST.items():
    if not h: continue
    for ax, metric in zip(axes, ["psnr", "ssim"]):
        pts = val_points(h, metric)
        if pts:
            xs, ys = zip(*pts)
            ax.plot(ep_axis(xs), ys, marker="s", label=name)
axes[0].set_title("Validation PSNR (DIV2K-Val crop 256)")
axes[1].set_title("Validation SSIM (DIV2K-Val crop 256)")
for ax in axes: ax.set_xlabel("image-epoch tương đương"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Đánh giá và bảng so sánh

Đánh giá cả 4 model đã train + (tùy chọn) SwinIR official, gom vào một bảng. Mỗi
dòng là một cặp (model, dataset), mỗi cột là một metric. Nếu chưa tải weights
SwinIR official, dòng đó sẽ được bỏ qua kèm hướng dẫn.

In [ ]:
import pandas as pd

rows = []

# 4 model tự train: dùng best.pth trong experiments/
for name, cfg_path in TRAIN_MODELS.items():
    print(f"\n========== EVAL {name} ==========")
    for ds_name, scores in eval_one(cfg_path, limit=EVAL_LIMIT).items():
        rows.append({"model": name, "dataset": ds_name, **scores})

# mốc SOTA tùy chọn: dùng weights chính thức
for name, (cfg_path, ckpt) in EVAL_ONLY_MODELS.items():
    if not Path(ckpt).exists():
        print(f"[bỏ qua {name}] chưa có weights: {ckpt}")
        print("  tải bằng: bash scripts/download_weights.sh")
        continue
    print(f"\n========== EVAL {name} ==========")
    for ds_name, scores in eval_one(cfg_path, checkpoint=ckpt, limit=EVAL_LIMIT).items():
        rows.append({"model": name, "dataset": ds_name, **scores})

df = pd.DataFrame(rows).set_index(["model", "dataset"]).round(4)
df

## Ghi chú

- Hướng tốt của metric: PSNR, SSIM, MUSIQ, CLIPIQA càng **cao** càng tốt; LPIPS,
  NIQE càng **thấp** càng tốt.
- Cả 4 model train cùng ngân sách 20 epoch và eval cùng DIV2K-Val crop 256 nên so
  sánh với nhau công bằng. Vì batch_size khác nhau, số lần cập nhật khác nhau:
  SRCNN/SRGAN ~520, SwinIR ~8.600, SR3 ~23.000 — SRCNN/SRGAN có thể còn undertrained.
- **SwinIR train ở đây là bản light (~0.9M params)** để vừa ngân sách; **SwinIR
  (official)** là bản M (~11.9M) với weights tác giả, chỉ là mốc SOTA tham chiếu,
  không cùng điều kiện. Muốn có nó thì `bash scripts/download_weights.sh` trước.
- Kiểm tra nhanh pipeline: đặt `EPOCHS_OVERRIDE = 3`, `EVAL_LIMIT = 10` ở mục Cấu
  hình. Số liệu khi đó chưa phải kết quả tốt.
- Lần đầu eval, MUSIQ/CLIPIQA sẽ tải weights về (cần mạng).